In [3]:
!pip -q install unsloth
!sudo apt install poppler-utils
!pip -q install pdf2image
!sudo apt install tesseract-ocr
!pip -q install pytesseract

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libpoppler-cpp0v5 libpoppler-glib8 libpoppler118
The following packages will be upgraded:
  libpoppler-cpp0v5 libpoppler-glib8 libpoppler118 poppler-utils
4 upgraded, 0 newly installed, 0 to remove and 395 not upgraded.
Need to get 1,430 kB of archives.
After this operation, 4,096 B of additional disk space will be used.
Do you want to continue? [Y/n] ^C

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 399 not upgraded.


In [4]:
import unsloth
import os
from unsloth import FastVisionModel
import torch
from datasets import load_dataset
from transformers import TextStreamer
from unsloth import is_bf16_supported
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

In [5]:
!pip install -q huggingface_hub

from huggingface_hub import login

# from google.colab import userdata
# hf_token = ''
hf_token = ''

login(token=hf_token)

In [6]:
from PIL import Image
def preprocess_image(img: Image.Image) -> Image.Image:
    # Grayscale & binarize
    gray = img.convert("L")
    binarized = gray.point(lambda x: 0 if x < 128 else 255, "1")
    return binarized

In [7]:
# Step 1: OCR function
import pytesseract
def extract_text_from_image(image):
    image = preprocess_image(image)
    text = pytesseract.image_to_string(image)
    return text.strip()

In [8]:
import re
def get_transactions_prompt(img):
  ocr_text = extract_text_from_image(img)
  ocr_text = re.sub(r'\n\s*', '\n', ocr_text)
  ocr_text = re.sub(r'\n+', '\n', ocr_text) 

  return f"""You are an expert bank statement analyzer.Given the OCR-extracted text from a bank statement image.return data in form of list of dictionary ,in form of list of JSON objects. please no explanation

Extract all the transactions from the text in the form of a list of dictionaries with the following keys (type of the field is defined in the bracket):

- TXN_DATE (Type: Date, Format: YYYY-MM-DD)
- TXN_DESC (Type: String)
- CHEQUE_REF_NO (Type: String. Only include alphanumeric cheque or reference numbers. Do NOT include monetary amounts.)
- WITHDRAWAL_AMT (Type: Float. Only include if money is withdrawn.may be mentioned as debit in statement. Set as null if not applicable.)
- DEPOSIT_AMT (Type: Float. Only include if money is deposited.may be mentioned as credit in statement. Set as null if not applicable.)
- BALANCE_AMT (Type: Float)
OCR Text:
\"\"\"
{ocr_text}
\"\"\"
any Transaction amount(WITHDRAWAL_AMT/DEPOSIT_AMT) has to be Positive , If something is missing or unexpected , back calculate with math to match the BALANCE_AMT. return only JSON , nothing else, If you do not get any data just return ,but never return explanation or code snippet,only return json.
JSON Output:
"""

In [9]:
import json
dataset = load_dataset("siddharthaspr/bankstatementdata", split = "train")


Generating train split: 100%|█████████| 117/117 [00:00<00:00, 135.21 examples/s]


In [ ]:
# import json
# dataset = load_dataset("tusharshah2006/bank_statements_transactions", split = "train")


def convert_to_conversation(sample):
    conversation = [
        { "role": "user",
          "content" : [
            {"type" : "text",  "text"  : get_transactions_prompt(sample["image"])},]
            # {"type" : "image", "image" : sample["image"]} ]
        },
        { "role" : "assistant",
          "content" : [
            {"type" : "text",  "text"  : sample['transactions'].replace('\n', '')   } ]
        },
    ]
    return { "messages" : conversation }
pass

converted_dataset = [convert_to_conversation(sample) for sample in dataset]


In [ ]:
from datasets import Dataset

new_dataset = Dataset.from_list(converted_dataset)


In [ ]:
def format_prompt(sample):
    user_msg = sample["messages"][0]["content"][0]["text"]
    assistant_msg = sample["messages"][1]["content"][0]["text"]
    prompt = f"<|user|>\n{user_msg}\n<|end|>\n<|assistant|>\n"
    return {
        "prompt": prompt,
        "response": assistant_msg,
        "full_text": prompt + assistant_msg
    }

new_dataset = new_dataset.map(format_prompt)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-Instruct-v0.2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token  # for safety

def tokenize(batch):
    full_texts = batch["full_text"]
    outputs = tokenizer(
        full_texts,
        truncation=True,
        padding=True,
        max_length=2048,
        return_tensors=None
    )
    outputs["labels"] = outputs["input_ids"].copy()
    return outputs


tokenized_dataset = new_dataset.map(tokenize,batched=True)


In [ ]:
from transformers import AutoModelForCausalLM
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    "mistralai/Mistral-7B-Instruct-v0.2",
    load_in_4bit=True,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)


In [ ]:
from transformers import TrainingArguments, Trainer


training_args = TrainingArguments(
    output_dir="outputs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=20,  # 👈 Run 20 epochs
    learning_rate=2e-4,
    weight_decay=0.01,
    logging_steps=10,
    save_steps=100,
    fp16=True,  # Set this depending on your GPU
    evaluation_strategy="no",
    report_to="none",
)


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    tokenizer=tokenizer
)


In [ ]:
trainer.train()

In [ ]:
from huggingface_hub import HfApi

repo_id = "siddharthaspr/Mistral-7B-Instruct-v0.2-bankstatement"

HfApi().create_repo(repo_id=repo_id, exist_ok=True, token=hf_token)


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

model.push_to_hub(repo_id, use_auth_token=hf_token)
tokenizer.push_to_hub(repo_id, use_auth_token=hf_token)
